In [ ]:
import os.path
import re
import sys

sys.path.extend([".", ".."])
import time
from tqdm import tqdm
import json
import cProfile

# from transformers import AutoTokenizer
from utils.java_parser import parse_fields_from_class_code
from data.configuration import code_base
from utils.prompt_formats.new_prompt_formatter import PromptFormatter

In [ ]:
profiler = cProfile.Profile()
focal_method_key = "source:source_method_code_format"
focal_method_name_key = "source:source_method_name"
focal_method_signature_key = "source:source_method_signature"
focal_method_docstring_key = "source:source_method_docstring"
focal_method_param_key = "source:source_method_parameters"
focal_method_return_type_key = "source:source_return_type"
focal_class_other_methods_key = "source:source_other_method_signature"
focal_class_fields_key = "source_class_fields"
focal_class_constructor_key = "content:source_class_constructors"
focal_class_name_key = "content:source_class_name"
focal_class_code_key = "content:source_class_code_format"
focal_class_docstring_key = "content:source_class_docstring"
source_method_type_constructor_key = "content:parameter_class_constructors"
source_method_return_type_constructor_key = "content:return_type_class_constructors"
source_method_parameter_key = "content:parameter_list"
parameter_classes_key = "content:parameter_class_signature"
source_method_signature_key = "source:source_method_signature"
source_class_imports_key = "content:source_class_code_imports"
project_name_key = 'extra:project_name'

In [ ]:
formatter = PromptFormatter()

In [ ]:
model_name = 'chatgpt'
tgt_model = 'GPT-O3-MINI'
format = 'comment'
strategy = 'generation'
version = 'fixed'
ignore_feature = "full"
min_test_num = 10

In [ ]:
# file_post_fix = f'unified_invoked'
# file_post_fix = f'unified_invoked_docstring_prompt'
# file_post_fix = f'unified_invoked_docstring_code_prompt'
# file_post_fix = f'unified_invoked_docstring_cot_prompt'
# file_post_fix = f'unified_invoked_docstring_cot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_cot_prompt_min_test_num_{min_test_num}_err_msg_prompt'
# file_post_fix = f'unified_invoked_docstring_cot_few_shot_prompt'
# file_post_fix = f'unified_invoked_code_cot_few_shot_prompt_min_test_num_{min_test_num}'
file_post_fix = f'unified_invoked_docstring_cot_few_shot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_few_shot_prompt_min_test_num_{min_test_num}'

In [ ]:
# Load the data
data = []
source_data_file = os.path.join(code_base, f"data/prompts/{tgt_model}_{version}_source_data_unified_invoked_with_docstring.jsonl")
if os.path.exists(source_data_file):
    with open(source_data_file, "r") as f:
        for line in f:
            data.append(json.loads(line))

In [ ]:
len(data)

In [ ]:
example_instance = data[0]

In [ ]:
print(example_instance[focal_method_key])

In [ ]:
example_instance[focal_method_return_type_key]

In [ ]:
example_new_inst = {}
bug_id = example_instance["extra:project_name"]
example_new_inst["focal_method_invoked"] = example_instance["focal_method_invoked"]
example_new_inst["id"] = bug_id
example_new_inst["strategy"] = strategy
example_new_inst["format"] = format
example_new_inst["ablation"] = ignore_feature
example_new_inst["focal_method"] = example_instance[focal_method_key]
is_public, example_new_inst['content'] = formatter.apply_format_with_docstring(
    example_instance,
    model_name,
    strategy=strategy,
    formatting=format,
    ignore_feature=ignore_feature,
    project_name=bug_id,
    min_test_num=min_test_num,
    with_examples=True,
    # with_docstring=True,
    # with_code=True,
    with_cot=True,
)
example_new_inst["class_name"] = bug_id
example_new_inst["method_signature"] = example_instance['source:source_method_signature']
example_new_inst["is_public"] = "1" if is_public else "0"
example_new_inst["role"] = "user"


In [ ]:
print(example_new_inst['content'])

In [ ]:
new_instance_data = []

In [ ]:
pbar = tqdm(data, total=len(data))
for idx, instance in enumerate(pbar):
    new_instance = {}
    bug_id = instance["extra:project_name"]
    new_instance["focal_method_invoked"] = instance["focal_method_invoked"]
    new_instance["id"] = bug_id
    new_instance["strategy"] = strategy
    new_instance["format"] = format
    new_instance["ablation"] = ignore_feature
    new_instance["focal_method"] = instance[focal_method_key]
    is_public, new_instance['content'] = formatter.apply_format_with_docstring(
        instance,
        model_name,
        strategy=strategy,
        formatting=format,
        ignore_feature=ignore_feature,
        project_name=bug_id,
        min_test_num=min_test_num,
        with_examples=True,
        # with_docstring=True,
        # with_code=True,
        with_cot=True,
    )

    new_instance["class_name"] = bug_id
    new_instance["method_signature"] = instance['source:source_method_signature']
    new_instance["is_public"] = "1" if is_public else "0"
    new_instance["role"] = "user"
    new_instance_data.append(new_instance)

In [ ]:
print(new_instance_data[0]['content'])

In [ ]:
output_file = os.path.join(
    code_base,
    "data/prompts/rq1/"
    + tgt_model
    + f"_{format}_{strategy}_{ignore_feature}_{version}_{file_post_fix}.jsonl",
    # + f"_{format}_{strategy}_{ignore_feature}_buggy_unified_invoked_docstring_cot_prompt.jsonl",
)

In [ ]:
output_file

In [ ]:
with open(output_file, "w") as f:
    for instance in new_instance_data:
        f.write(json.dumps(instance) + "\n")